## 00\_citiz\_lovelo

# 00 · Stations Citiz et Lovélo (flux GBFS)

Collecte des stations d'autopartage Citiz (15 réseaux) et des stations de vélos en libre-service Lovélo (Métropole Rouen Normandie) à partir de leurs flux GBFS publics.

1. Téléchargement de la position et de la capacité de chaque station.
2. Contrôle de périmètre : absence de station Citiz dans la zone de Rouen, présence dans l'Eurométropole de Strasbourg.
3. Relevé horodaté du nombre de véhicules disponibles par station Citiz. Une exécution planifiée accumule ces relevés dans `releves/`.

**Sorties** (`Files/brut/gbfs/`) : `stations_citiz.csv`, `stations_lovelo.csv`, `types_vehicules_citiz.csv`, `releves/releve_AAAAMMJJ_HHMM.csv`.

In [ ]:
# 1. Dossiers de travail
import os, csv, json, time, urllib.request
from datetime import datetime, timezone

BASE = "/lakehouse/default/Files"
if not os.path.isdir(BASE):
    raise RuntimeError(
        "Lakehouse par défaut non attaché : attacher le Lakehouse mobilite au notebook."
    )
DOSSIER = os.path.join(BASE, "brut", "gbfs")
DOSSIER_RELEVES = os.path.join(DOSSIER, "releves")
os.makedirs(DOSSIER_RELEVES, exist_ok=True)
print("Dossier de sortie :", DOSSIER)

Dossier de sortie : /lakehouse/default/Files/brut/gbfs


In [ ]:
# 2. Les flux GBFS 3.0 (adresses du catalogue MobilityData et de transport.data.gouv.fr)
CITIZ = {
    "citiz_grand_est": "https://backend.citiz.fr/public/provider/1/gbfs/v3.0/gbfs.json",
    "citiz_bfc": "https://backend.citiz.fr/public/provider/2/gbfs/v3.0/gbfs.json",
    "citiz_metropole_lyon": "https://backend.citiz.fr/public/provider/3/gbfs/v3.0/gbfs.json",
    "citiz_gironde": "https://backend.citiz.fr/public/provider/4/gbfs/v3.0/gbfs.json",
    "citiz_alpes_loire": "https://backend.citiz.fr/public/provider/5/gbfs/v3.0/gbfs.json",
    "citiz_provence": "https://backend.citiz.fr/public/provider/6/gbfs/v3.0/gbfs.json",
    "citiz_angers": "https://backend.citiz.fr/public/provider/7/gbfs/v3.0/gbfs.json",
    "citiz_occitanie": "https://backend.citiz.fr/public/provider/8/gbfs/v3.0/gbfs.json",
    "citiz_grand_poitiers": "https://backend.citiz.fr/public/provider/9/gbfs/v3.0/gbfs.json",
    "citiz_developpement": "https://backend.citiz.fr/public/provider/12/gbfs/v3.0/gbfs.json",
    "citiz_nantes": "https://backend.citiz.fr/public/provider/15/gbfs/v3.0/gbfs.json",
    "citiz_lille_arras": "https://backend.citiz.fr/public/provider/16/gbfs/v3.0/gbfs.json",
    "citiz_la_rochelle": "https://backend.citiz.fr/public/provider/18/gbfs/v3.0/gbfs.json",
    "citiz_rennes_metropole": "https://backend.citiz.fr/public/provider/19/gbfs/v3.0/gbfs.json",
    "citiz_aupa": "https://backend.citiz.fr/public/provider/20/gbfs/v3.0/gbfs.json",
}
LOVELO = {
    "lovelo_rouen": "https://gbfs.urbansharing.com/lovelolibreservice.fr/gbfs.json",
}

# Rectangles approximatifs (lat_min, lat_max, lon_min, lon_max) pour les vérifications
ZONES = {
    "Rouen": (49.25, 49.58, 0.80, 1.32),
    "Strasbourg": (48.45, 48.72, 7.60, 7.90),
}

In [ ]:
# 3. Fonctions utilitaires
ENTETES = {"User-Agent": "hackathon-mobilite (releve GBFS, Microsoft Fabric)"}

def lire_json(url, essais=3):
    """Télécharge une adresse et renvoie le JSON. Réessaie en cas d'échec."""
    erreur = None
    for tentative in range(essais):
        try:
            requete = urllib.request.Request(url, headers=ENTETES)
            with urllib.request.urlopen(requete, timeout=30) as reponse:
                return json.loads(reponse.read().decode("utf-8"))
        except Exception as e:
            erreur = e
            time.sleep(2 * (tentative + 1))
    raise erreur

def variantes(url):
    """Les flux Citiz existent sous deux formes d'adresse : .../gbfs/v3.0/ (la bonne) et .../gbfs/3.0/."""
    if "/gbfs/v3.0/" in url:
        return [url, url.replace("/gbfs/v3.0/", "/gbfs/3.0/")]
    return [url]

def urls_des_flux(url_gbfs):
    """Renvoie {nom_du_flux: adresse} à partir du sommaire gbfs.json (GBFS 2 ou 3)."""
    data, derniere_erreur = None, None
    for url in variantes(url_gbfs):
        try:
            data = lire_json(url).get("data", {})
            break
        except Exception as e:
            derniere_erreur = e
    if data is None:
        raise derniere_erreur
    if "feeds" in data:                 # GBFS 3 : une liste directe
        flux = data["feeds"]
    else:                               # GBFS 2 : une liste par langue
        langue = data.get("fr") or data.get("en") or next(iter(data.values()), {})
        flux = langue.get("feeds", [])
    return {f["name"]: f["url"] for f in flux if "name" in f and "url" in f}

def texte(valeur):
    """Un nom est un texte en GBFS 2, une liste de traductions en GBFS 3."""
    if isinstance(valeur, list):
        for t in valeur:
            if str(t.get("language", "")).startswith("fr"):
                return t.get("text", "")
        return valeur[0].get("text", "") if valeur else ""
    return "" if valeur is None else valeur

def ecrire_csv(chemin, lignes, champs):
    with open(chemin, "w", newline="", encoding="utf-8") as f:
        ecrivain = csv.DictWriter(f, fieldnames=champs)
        ecrivain.writeheader()
        ecrivain.writerows(lignes)

def dans_zone(station, zone):
    lat_min, lat_max, lon_min, lon_max = ZONES[zone]
    try:
        lat, lon = float(station["lat"]), float(station["lon"])
    except (TypeError, ValueError):
        return False
    return lat_min <= lat <= lat_max and lon_min <= lon <= lon_max

In [ ]:
# 4. Télécharger les stations de chaque réseau
CHAMPS_STATIONS = ["reseau", "station_id", "nom", "lat", "lon", "capacite", "adresse"]
CHAMPS_TYPES = ["reseau", "vehicle_type_id", "nom", "form_factor", "propulsion_type", "max_range_meters"]

def telecharger_stations(reseaux):
    flux_par_reseau, stations, types = {}, [], []
    for reseau, url in reseaux.items():
        try:
            flux = urls_des_flux(url)
        except Exception as e:
            print(f"  [{reseau}] sommaire gbfs.json inaccessible : {e}")
            continue
        flux_par_reseau[reseau] = flux
        n = 0
        if "station_information" in flux:
            try:
                for s in lire_json(flux["station_information"])["data"]["stations"]:
                    stations.append({
                        "reseau": reseau,
                        "station_id": s.get("station_id", ""),
                        "nom": texte(s.get("name")),
                        "lat": s.get("lat", ""),
                        "lon": s.get("lon", ""),
                        "capacite": s.get("capacity", ""),
                        "adresse": s.get("address", ""),
                    })
                    n += 1
            except Exception as e:
                print(f"  [{reseau}] stations illisibles : {e}")
        if "vehicle_types" in flux:
            try:
                for t in lire_json(flux["vehicle_types"])["data"]["vehicle_types"]:
                    types.append({
                        "reseau": reseau,
                        "vehicle_type_id": t.get("vehicle_type_id", ""),
                        "nom": texte(t.get("name")),
                        "form_factor": t.get("form_factor", ""),
                        "propulsion_type": t.get("propulsion_type", ""),
                        "max_range_meters": t.get("max_range_meters", ""),
                    })
            except Exception as e:
                print(f"  [{reseau}] types de véhicules illisibles : {e}")
        print(f"  [{reseau}] {n} stations")
    return flux_par_reseau, stations, types

print("Citiz :")
flux_citiz, stations_citiz, types_citiz = telecharger_stations(CITIZ)
print("Lovélo :")
flux_lovelo, stations_lovelo, _ = telecharger_stations(LOVELO)

if not stations_citiz:
    raise RuntimeError("Aucune station Citiz récupérée : flux GBFS inaccessibles.")
ecrire_csv(os.path.join(DOSSIER, "stations_citiz.csv"), stations_citiz, CHAMPS_STATIONS)
ecrire_csv(os.path.join(DOSSIER, "types_vehicules_citiz.csv"), types_citiz, CHAMPS_TYPES)
if stations_lovelo:
    ecrire_csv(os.path.join(DOSSIER, "stations_lovelo.csv"), stations_lovelo, CHAMPS_STATIONS)
print(f"\nTotal : {len(stations_citiz)} stations Citiz sur {len(flux_citiz)} réseaux, "
      f"{len(stations_lovelo)} stations Lovélo.")

Citiz :
  [citiz_grand_est] 290 stations
  [citiz_bfc] 60 stations
  [citiz_metropole_lyon] 200 stations
  [citiz_gironde] 100 stations
  [citiz_alpes_loire] 412 stations
  [citiz_provence] 97 stations
  [citiz_angers] 28 stations
  [citiz_occitanie] 142 stations
  [citiz_grand_poitiers] 21 stations
  [citiz_developpement] 64 stations
  [citiz_nantes] 23 stations
  [citiz_lille_arras] 80 stations
  [citiz_la_rochelle] 24 stations
  [citiz_rennes_metropole] 66 stations
  [citiz_aupa] 8 stations
Lovélo :
  [lovelo_rouen] 126 stations

Total : 1615 stations Citiz sur 15 réseaux, 126 stations Lovélo.


In [ ]:
# 5. Vérifications : Citiz doit être absent de Rouen et présent à Strasbourg
citiz_rouen = [s for s in stations_citiz if dans_zone(s, "Rouen")]
citiz_strasbourg = [s for s in stations_citiz if dans_zone(s, "Strasbourg")]
lovelo_rouen = [s for s in stations_lovelo if dans_zone(s, "Rouen")]

print(f"Stations Citiz dans la zone de Rouen : {len(citiz_rouen)}")
for s in citiz_rouen[:10]:
    print(f"   - {s['reseau']} | {s['nom']}")
if citiz_rouen:
    print("Stations Citiz détectées dans la zone de Rouen.")
else:
    print("Absence de Citiz à Rouen confirmée.")
print(f"Stations Citiz dans la zone de Strasbourg : {len(citiz_strasbourg)}")
print(f"Stations Lovélo dans la zone de Rouen : {len(lovelo_rouen)}")

Stations Citiz dans la zone de Rouen : 0
Absence de Citiz à Rouen confirmée.
Stations Citiz dans la zone de Strasbourg : 185
Stations Lovélo dans la zone de Rouen : 126


In [ ]:
# 6. Relevé des voitures disponibles (un fichier par exécution)
horodatage = datetime.now(timezone.utc)
lignes = []
for reseau, flux in flux_citiz.items():
    url = flux.get("station_status")
    if not url:
        continue
    try:
        for s in lire_json(url)["data"]["stations"]:
            lignes.append({
                "horodatage_utc": horodatage.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "reseau": reseau,
                "station_id": s.get("station_id", ""),
                "voitures_disponibles": s.get("num_vehicles_available", s.get("num_bikes_available", "")),
                "station_ouverte": s.get("is_renting", ""),
                "dernier_signal": s.get("last_reported", ""),
            })
    except Exception as e:
        print(f"  [{reseau}] relevé impossible : {e}")

chemin = os.path.join(DOSSIER_RELEVES, horodatage.strftime("releve_%Y%m%d_%H%M.csv"))
ecrire_csv(chemin, lignes, ["horodatage_utc", "reseau", "station_id", "voitures_disponibles",
                            "station_ouverte", "dernier_signal"])
print(f"{len(lignes)} stations relevées dans {chemin}")
print(f"Relevés accumulés : {len(os.listdir(DOSSIER_RELEVES))}")

1615 stations relevées dans /lakehouse/default/Files/brut/gbfs/releves/releve_20261021_0813.csv
Relevés accumulés : 2
